# Question 3.(g).iv and 3.(g).v: exploration

Notebook to explore the results of the battery case step by step. It is **not submitted** (the `notebooks/`
folder is removed before submission): everything used in the report comes from `python main.py --question Q3_battery`,
and this notebook calls the same functions.

1. (iv) Base case with and without the battery
2. (v) Sensitivity sweeps (the ones of `main.py`, plus a cell to try other values)
3. End of the day: our $\pi^{end}$ against the alternatives, and a fine sweep of $\pi^{end}$
4. Extended-horizon benchmark (35 h) and the value of the stored energy
5. Notes

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from main import Q3_BATTERY_SWEEPS, battery_run, end_of_day_alternatives, schedule_change
from src.data_loader import load_question
from src.model import FlexibleConsumerModel, solve_extended_horizon, terminal_value
from src.plotting import plot_battery_schedule, plot_battery_sensitivity

pd.set_option("display.precision", 3)
B = load_question("Q3_battery")      # with the battery
NB = load_question("Q3")             # the same consumer without the battery (3.(d))
p_imp = B.energy_price + B.import_tariff
eta_ch = B.battery_charging_efficiency
print(B.summary())

## 1. (iv) Base case

Solve with and without the battery. The value of the battery is the difference of the objectives.

In [ ]:
w = FlexibleConsumerModel(B).build().solve()
wo = FlexibleConsumerModel(NB).build().solve()
hw, hwo = w.hourly, wo.hourly

def disutility(h, d=B):
    return d.quadratic_disutility * ((h["load"] - h["reference_load"]) ** 2).sum()

summary = pd.DataFrame({
    "without battery (d)": [hwo["import"].sum(), hwo["export"].sum(), wo.meta["procurement_cost"], disutility(hwo),
                            0.0, wo.objective, wo.duals["min_energy"], hwo["load"].sum()],
    "with battery": [hw["import"].sum(), hw["export"].sum(), w.meta["procurement_cost"], disutility(hw),
                     w.meta["terminal_value"], w.objective, w.duals["min_energy"], hw["load"].sum()],
}, index=["import (kWh)", "export (kWh)", "procurement cost (DKK)", "disutility (DKK)",
          "pi_end (e_23 - e_init) (DKK)", "objective (DKK)", "mu^E (DKK/kWh)", "total load (kWh)"])
summary["difference"] = summary["with battery"] - summary["without battery (d)"]
print(f"Value of the battery: {w.objective - wo.objective:.3f} DKK/day,  pi_end = {w.meta['pi_end']:.3f} DKK/kWh")
print(f"Largest overlap of charging and discharging: {w.meta['overlap_max']:.1e} kWh/h")
summary

**Split of the value:** procurement saving + disutility saving + stored-energy term. The last term is negative
because the battery starts with 2 kWh and ends empty: those 2 kWh are paid at $\pi^{end}$.

In [ ]:
procurement_saving = wo.meta["procurement_cost"] - w.meta["procurement_cost"]
disutility_saving = disutility(hwo) - disutility(hw)
print(f"{procurement_saving:.3f} + {disutility_saving:.3f} + ({w.meta['terminal_value']:.3f}) = "
      f"{procurement_saving + disutility_saving + w.meta['terminal_value']:.3f}  (value of the battery)")
plot_battery_schedule(w, B, wo); plt.show()

**Hour by hour:** why does the load barely change? At an interior optimum
$\ell_t = \hat\ell_t + (\mu^E - \lambda_t)/(2c^Q)$, so the load only moves where $\lambda_t$ (dual of the balance)
or $\mu^E$ changes.

In [ ]:
hours = pd.DataFrame({
    "p_imp": p_imp, "ref": hw["reference_load"],
    "load without": hwo["load"], "load with": hw["load"], "d_load": hw["load"] - hwo["load"],
    "lambda without": hwo["dual_balance"], "lambda with": hw["dual_balance"],
    "charge": hw["charge"], "discharge": hw["discharge"], "soc": hw["soc"],
    "export without": hwo["export"], "export with": hw["export"],
})
print("Sum of |d_load| =", round(hours["d_load"].abs().sum(), 3), "kWh")
hours.round(3)

## 2. (v) Sensitivity sweeps

The sweeps of `main.py` (`Q3_BATTERY_SWEEPS`): one parameter at a time, the rest at the base case.
`battery_run` solves with and without the battery on the same data.

In [ ]:
rows = []
for name, sw in Q3_BATTERY_SWEEPS.items():
    for v in sw["values"]:
        r = battery_run(sw["apply"](B, v))
        rows.append(dict(parameter=name, value=v, **{k: x for k, x in r.items() if k != "hourly"}))
sens = pd.DataFrame(rows)
plot_battery_sensitivity(sens, {n: sw["label"] for n, sw in Q3_BATTERY_SWEEPS.items()}); plt.show()
sens.round(3)

**Try other values.** Change `name` and `values`, or build a scenario by hand with `src/scenarios.py`
(`scale_prices`, `shift_prices`, `roll_prices`, `set_tariffs`, `scale_pv`, `set_load_preferences`, `set_disutility`,
`set_battery`).

In [ ]:
name, values = "capacity", [0.5, 1, 3, 4, 6, 10]
pd.DataFrame([dict(value=v, **{k: x for k, x in battery_run(Q3_BATTERY_SWEEPS[name]["apply"](B, v)).items()
                               if k != "hourly"}) for v in values]).round(3)

## 3. End of the day

Every price-based alternative uses the same objective term, $+\pi^{end}(e_{23} - e^{init})$; only the number changes.

| alternative | $\pi^{end}$ |
|---|---|
| ours (i) | $\min\{\min_{night} p^{imp}_{D+1,t}/\eta^{ch},\ \eta^{dis}\max_t p^{imp}_{D+1,t}\}$ |
| trend | one-hour extrapolation: $p^{imp}_{23} + (p^{imp}_{23} - p^{imp}_{22})$ |
| last price | $p^{imp}_{23}$ |
| last price / $\eta^{ch}$ | $p^{imp}_{23}/\eta^{ch}$ (the cost of recharging in hour 23) |
| free end | $\pi^{end}=0$ |
| cyclic | $e_{23} \geq e^{init}$, no term |

"schedule_change" is the largest change of charging or discharging in any hour compared with ours (kWh/h).

In [ ]:
ref = battery_run(B)
rows = []
for name, args in end_of_day_alternatives(B).items():
    r = battery_run(B, **args)
    rows.append(dict(end=name, pi_end=r["pi_end"], final_soc=r["final_soc_kWh"], value=r["value_DKK"],
                     schedule_change=schedule_change(r["hourly"], ref["hourly"]), discharged=r["discharged_kWh"]))
ends = pd.DataFrame(rows).set_index("end")
print(f"p_imp_22 = {p_imp[22]:.2f}, p_imp_23 = {p_imp[23]:.2f}; recharging in hour 23 costs p_imp_23/eta_ch = {p_imp[23]/eta_ch:.3f}")
ends.round(3)

**Fine sweep of $\pi^{end}$**: where the decisions change (threshold) and how the value of the battery changes.
The dotted lines are the price-based alternatives of the table above.

In [ ]:
grid = np.round(np.arange(0.0, 2.81, 0.05), 2)
fine = []
for pe in grid:
    r = battery_run(B, pi_end=float(pe))
    fine.append(dict(pi_end=pe, final_soc=r["final_soc_kWh"], value=r["value_DKK"],
                     schedule_change=schedule_change(r["hourly"], ref["hourly"])))
fine = pd.DataFrame(fine)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharex=True)
for ax, col, lab in zip(axes, ["final_soc", "schedule_change", "value"],
                        ["$e_{23}$ (kWh)", "schedule change (kWh/h)", "value of the battery (DKK/day)"]):
    ax.plot(fine["pi_end"], fine[col], color="C0", lw=2)
    for name, x in ends["pi_end"].items():
        if name not in ("free end", "cyclic"):
            ax.axvline(x, color="grey", lw=0.8, ls=":")
    ax.set(xlabel=r"$\pi^{end}$ (DKK/kWh)", ylabel=lab); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"Decisions change from pi_end ≈ {fine.loc[fine['schedule_change'] > 0.05, 'pi_end'].min()} "
      "(main.py finds the exact thresholds on a 0.01 grid: 1.71 and 1.79)")

## 4. Extended-horizon benchmark (35 h)

The licence limits a QP to **200 variables**. The model of `src/model.py` has 7 variables per hour (load, PV,
import, export, charge, discharge, state of charge), so only 28 h would fit: up to 4 AM of D+1, before the morning peak.

`solve_extended_horizon()` (in `src/model.py`) writes the same problem in a compact form: the state of charge is an
expression, PV variables only exist in hours with PV, and the bounds are variable bounds. Day D plus hours 0-10 of D+1
(**35 h**, including the morning peak at hour 9) then fit in 196 variables. $E^{min}$ is imposed on day D only, and the
energy left at the end of the horizon has no value.

In [ ]:
ext = solve_extended_horizon(B)
print("variables:", ext["n_variables"], "(limit 200)")
print("e_23 in the benchmark:", ext["soc"][23].round(3), " | in our daily model:", hw["soc"].iloc[-1].round(3))
print("schedule change on day D:", round(schedule_change(ext, hw), 4), "kWh/h")

h = np.arange(len(ext["soc"]))
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.bar(h, ext["charge"], color="C2", alpha=0.6, label="charge (benchmark)")
ax.bar(h, -ext["discharge"], color="C3", alpha=0.6, label="discharge (benchmark)")
ax.plot(np.r_[-0.5, h + 0.5], np.r_[B.battery_initial_soc_kWh, ext["soc"]], color="C4", marker="o", ms=3,
        label="state of charge (benchmark)")
ax.plot(np.r_[-0.5, np.arange(24) + 0.5], np.r_[B.battery_initial_soc_kWh, hw["soc"]], color="k", ls="--",
        label="state of charge (daily model)")
ax.axvline(23.5, color="grey", lw=1); ax.text(23.7, 3.8, "D+1")
ax.set(xlabel="hour (24 = hour 0 of D+1)", ylabel="kWh"); ax.legend(fontsize=8, ncol=2); ax.grid(alpha=0.3)
plt.show()

**Value of the stored energy (water value).** How much is it worth to start D+1 with $x$ kWh in the battery? Solve
only hours 0-10 of D+1 starting from $x$ and compare with starting empty. The slope is the value of the marginal kWh;
our $\pi^{end}$ is a single straight line. The exact value of the battery replaces our term
$\pi^{end}(e_{23} - e^{init})$ by $W(e_{23}) - W(e^{init})$.

In [ ]:
xs = np.arange(0, B.battery_capacity_kWh + 1e-9, 0.25)
W = np.array([solve_extended_horizon(B, today=False, start_soc=float(x))["objective"] for x in xs])
W -= W[0]
slope = np.diff(W) / np.diff(xs)
pi_end = terminal_value(B)
e23, e0 = hw["soc"].iloc[-1], B.battery_initial_soc_kWh
exact = (w.objective - wo.objective) - pi_end * (e23 - e0) + np.interp(e23, xs, W) - np.interp(e0, xs, W)
print(f"Exact value of the battery: {exact:.3f} DKK/day (ours: {w.objective - wo.objective:.3f})")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.6))
a1.plot(xs, W, color="C0", lw=2, marker="o", ms=4, label="exact value (benchmark)")
a1.plot(xs, pi_end * xs, color="k", ls="--", label=fr"our $\pi^{{end}}$ = {pi_end:.2f}")
a1.set(xlabel="energy at the start of D+1 (kWh)", ylabel="value (DKK)"); a1.legend(); a1.grid(alpha=0.3)
a2.step(xs[1:], slope, where="pre", color="C0", lw=2, label="value of the marginal kWh")
a2.axhline(pi_end, color="k", ls="--", label=r"our $\pi^{end}$")
a2.axhline(1.71, color="C3", ls=":", label="threshold where the decisions change")
a2.set(xlabel="energy at the start of D+1 (kWh)", ylabel="DKK/kWh", ylim=(1.3, 1.9)); a2.legend(fontsize=8)
a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("slopes:", slope.round(3), "| mean over 4 kWh:", round(W[-1] / 4, 3))

## 5. Notes

- What convinces me:
- What I want to change in the text of (iv)/(v):
- What to add to the report (and therefore to `main.py`):